# Notebook 01 (v2) — BRFSS 2023 Data Cleaning

**v2 = revised after reviewer comments. The original `01_cleaning.ipynb` is unchanged.**

### What changed from v1
| # | Change | Reviewer comment |
|---|---|---|
| 1 | State, race and survey-design columns are kept as `meta_*` side columns (never model features) | #7 (survey weights), #9 (fairness) |
| 2 | Duplicate "drop columns" cell removed; per-group drop counts printed | #11 (numerical consistency) |
| 3 | Hypertension and diabetes recoded with the CDC definitions; the invented `avg_glucose` proxy is replaced by an ordinal `diabetes_status` | #11, #17 |
| 4 | **No outcome-dependent row filtering.** All rows with a known target are kept; a `meta_missing_pct` column is saved so any filtering is done later, identically for both classes | #2 (major) |
| 5 | A cleaning flow table (rows / columns at each step) is saved for the paper | #11, #12 |

### What this notebook does
- Loads the raw BRFSS 2023 dataset
- Removes administrative and calculated duplicate columns
- Fixes coding issues (special missing codes, binary remapping)
- Saves the cleaned dataset (all respondents with a known stroke answer)

**Source** — Dataset: BRFSS 2023. Codebook: USCODE23_LLCP_021924.HTML (official CDC 2023 codebook)

**Output** — `data/processed/brfss_cleaned_v2.csv`, `outputs/cleaning_flow_v2.csv`

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR      = '/content/drive/MyDrive/stroke_prediction'
RAW_PATH      = os.path.join(BASE_DIR, 'data', 'raw', 'LLCP2023.XPT')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
OUTPUTS_DIR   = os.path.join(BASE_DIR, 'outputs')
# v2: new file names — v1 outputs are NOT overwritten
CLEANED_PATH  = os.path.join(PROCESSED_DIR, 'brfss_cleaned_v2.csv')
FLOW_PATH     = os.path.join(OUTPUTS_DIR, 'cleaning_flow_v2.csv')

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("Project root  :", BASE_DIR)
print("Raw file      :", RAW_PATH)
print("Output file   :", CLEANED_PATH)

if os.path.exists(RAW_PATH):
    print("\n✓ Raw BRFSS file found — ready to load")
else:
    print("\n✗ Raw file NOT found — check the path above")

In [ ]:
# pyreadstat is needed to read SAS XPT files
# BRFSS comes in SAS XPT format which pandas cannot read directly
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyreadstat', '-q'])

import pyreadstat
print("pyreadstat version:", pyreadstat.__version__)
print("✓ Ready to read XPT files")

In [ ]:
print("Loading BRFSS 2023 dataset...")
print("This may take 1-2 minutes — file is large")
print()

df, meta = pyreadstat.read_xport(RAW_PATH, encoding='latin1')

print(f"✓ Dataset loaded successfully")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")
print(f"   Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# v2: record every step for the cleaning flow table (Change 5)
flow = []
def log_step(step, note=''):
    feature_cols = [c for c in df.columns if not c.startswith('meta_')]
    flow.append({
        'step'           : step,
        'rows'           : len(df),
        'columns_total'  : df.shape[1],
        'columns_no_meta': len(feature_cols),
        'stroke_1'       : int((df['stroke'] == 1).sum()) if 'stroke' in df.columns else None,
        'note'           : note,
    })
    print(f"[flow] {step}: {len(df):,} rows x {df.shape[1]} columns")

log_step('Raw BRFSS 2023 file')

In [ ]:
# Get a first look at the data
print("=== FIRST 3 ROWS ===")
print(df.head(3))
print()

print("=== DATA TYPES (first 20 columns) ===")
print(df.dtypes.head(20))
print()

# Find the extra columns not in the codebook
codebook_vars = [
    '_STATE','FMONTH','IDATE','IMONTH','IDAY','IYEAR','DISPCODE','SEQNO','_PSU',
    'CTELENM1','PVTRESD1','COLGHOUS','STATERE1','CELPHON1','LADULT1','NUMADULT',
    'RESPSLC1','LANDSEX2','SAFETIME','CTELNUM1','CELLFON5','CADULT1','CELLSEX2',
    'PVTRESD3','CCLGHOUS','CSTATE1','LANDLINE','HHADULT','SEXVAR','GENHLTH',
    'PHYSHLTH','MENTHLTH','POORHLTH','PRIMINS1','PERSDOC3','MEDCOST1','CHECKUP1',
    'EXERANY2','EXRACT12','EXEROFT1','EXERHMM1','EXRACT22','EXEROFT2','EXERHMM2',
    'STRENGTH','BPHIGH6','BPMEDS1','CHOLCHK3','TOLDHI3','CHOLMED3','CVDINFR4',
    'CVDCRHD4','CVDSTRK3','ASTHMA3','ASTHNOW','CHCSCNC1','CHCOCNC1','CHCCOPD3',
    'ADDEPEV3','CHCKDNY2','HAVARTH4','DIABETE4','DIABAGE4','MARITAL','EDUCA',
    'RENTHOM1','NUMHHOL4','NUMPHON4','CPDEMO1C','VETERAN3','EMPLOY1','CHILDREN',
    'INCOME3','PREGNANT','WEIGHT2','HEIGHT3','DEAF','BLIND','DECIDE','DIFFWALK',
    'DIFFDRES','DIFFALON','FALL12MN','FALLINJ5','SMOKE100','SMOKDAY2','USENOW3',
    'ECIGNOW2','ALCDAY4','AVEDRNK3','DRNK3GE5','MAXDRNKS','FLUSHOT7','FLSHTMY3',
    'PNEUVAC4','SHINGLE2','HIVTST7','HIVTSTD3','SEATBELT','DRNKDRI2','COVIDPO1',
    'COVIDSM1','COVIDACT','PDIABTS1','PREDIAB2','DIABTYPE','INSULIN1','CHKHEMO3',
    'EYEEXAM1','DIABEYE1','DIABEDU1','FEETSORE','ARTHEXER','ARTHEDU','LMTJOIN3',
    'ARTHDIS2','JOINPAI2','LCSFIRST','LCSLAST','LCSNUMCG','LCSCTSC1','LCSSCNCR',
    'LCSCTWHN','HADMAM','HOWLONG','CERVSCRN','CRVCLCNC','CRVCLPAP','CRVCLHPV',
    'HADHYST2','PSATEST1','PSATIME1','PCPSARS2','PSASUGS1','PCSTALK2','HADSIGM4',
    'COLNSIGM','COLNTES1','SIGMTES1','LASTSIG4','COLNCNCR','VIRCOLO1','VCLNTES2',
    'SMALSTOL','STOLTEST','STOOLDN2','BLDSTFIT','SDNATES1','CNCRDIFF','CNCRAGE',
    'CNCRTYP2','CSRVTRT3','CSRVDOC1','CSRVSUM','CSRVRTRN','CSRVINST','CSRVINSR',
    'CSRVDEIN','CSRVCLIN','CSRVPAIN','CSRVCTL2','INDORTAN','NUMBURN3','SUNPRTCT',
    'WKDAYOUT','WKENDOUT','CIMEMLO1','CDWORRY','CDDISCU1','CDHOUS1','CDSOCIA1',
    'CAREGIV1','CRGVREL4','CRGVLNG1','CRGVHRS1','CRGVPRB3','CRGVALZD','CRGVPER1',
    'CRGVHOU1','CRGVEXPT','LASTSMK2','STOPSMK2','MENTCIGS','MENTECIG','HEATTBCO',
    'FIREARM5','GUNLOAD','LOADULK2','HASYMP1','HASYMP2','HASYMP3','HASYMP4',
    'HASYMP5','HASYMP6','STRSYMP1','STRSYMP2','STRSYMP3','STRSYMP4','STRSYMP5',
    'STRSYMP6','FIRSTAID','ASPIRIN','SOMALE','SOFEMALE','MARIJAN1','MARJSMOK',
    'MARJEAT','MARJVAPE','MARJDAB','MARJOTHR','ACEDEPRS','ACEDRINK','ACEDRUGS',
    'ACEPRISN','ACEDIVRC','ACEPUNCH','ACEHURT1','ACESWEAR','ACETOUCH','ACETTHEM',
    'ACEHVSEX','ACEADSAF','ACEADNED','IMFVPLA4','HPVADVC4','HPVADSHT','TETANUS1',
    'COVIDVA1','COVACGE1','COVIDNU2','LSATISFY','EMTSUPRT','SDLONELY','SDHEMPLY',
    'FOODSTMP','SDHFOOD1','SDHBILLS','SDHUTILS','SDHTRNSP','SDHSTRE1','RRCLASS3',
    'RRCOGNT2','RRTREAT','RRATWRK2','RRHCARE4','RRPHYSM2','RCSBORG1','RCSRLTN2',
    'CASTHDX2','CASTHNO2','QSTVER','QSTLANG','_METSTAT','_URBSTAT','MSCODE',
    '_STSTR','_STRWT','_RAWRAKE','_WT2RAKE','_IMPRACE','_CHISPNC','_CRACE1',
    'CAGEG','_CLLCPWT','_DUALUSE','_DUALCOR','_LLCPWT2','_LLCPWT','_RFHLTH',
    '_PHYS14D','_MENT14D','_HLTHPL1','_HCVU653','_TOTINDA','METVL12_','METVL22_',
    'MAXVO21_','FC601_','ACTIN13_','ACTIN23_','PADUR1_','PADUR2_','PAFREQ1_',
    'PAFREQ2_','_MINAC12','_MINAC22','STRFREQ_','PAMISS3_','PAMIN13_','PAMIN23_',
    'PA3MIN_','PAVIG13_','PAVIG23_','PA3VIGM_','_PACAT3','_PAINDX3','_PA150R4',
    '_PA300R4','_PA30023','_PASTRNG','_PAREC3','_PASTAE3','_RFHYPE6','_CHOLCH3',
    '_RFCHOL3','_MICHD','_LTASTH1','_CASTHM1','_ASTHMS1','_DRDXAR2','_MRACE1',
    '_HISPANC','_RACE','_RACEG21','_RACEGR3','_RACEPRV','_SEX','_AGEG5YR',
    '_AGE65YR','_AGE80','_AGE_G','HTIN4','HTM4','WTKG3','_BMI5','_BMI5CAT',
    '_RFBMI5','_CHLDCNT','_EDUCAG','_INCOMG1','_SMOKER3','_RFSMOK3','_CURECI2',
    'DRNKANY6','DROCDY4_','_RFBING6','_DRNKWK2','_RFDRHV8','_FLSHOT7','_PNEUMO3',
    '_AIDTST4','_RFSEAT2','_RFSEAT3','_DRNKDRV'
]

extra_cols = [c for c in df.columns if c not in codebook_vars]
print(f"=== EXTRA COLUMNS NOT IN CODEBOOK ({len(extra_cols)}) ===")
for col in extra_cols:
    print(f"  {col}")

In [ ]:
# Codebook variables that are documented but absent from the actual file
in_codebook_not_in_file = set(codebook_vars) - set(df.columns)
print(f"Variables documented in codebook but ABSENT from file: {len(in_codebook_not_in_file)}")
for v in sorted(in_codebook_not_in_file):
    print(f"  {v}")

# ── BRFSS 2023 File vs Codebook ───────────────────────────────────────────
# Codebook documents 344 variables: 343 present in file, 1 absent (RCSBORG1)
# 7 undocumented variables present in file:
#   LNDSXBRT, CELSXBRT, BIRTHSEX, TRNSGNDR, USEMRJN4, RCSGEND1, RCSXBRTH
# Total columns in file : 343 + 7 = 350

### Change 1 — Keep side-information columns (not model features)

State, survey weight, sampling design and race are copied into `meta_*` columns **before** the drop step.
They are needed later for subgroup (fairness) analysis, survey-weighted analysis and for detecting
optional-module questions. **Every later notebook must drop `meta_*` columns before training.**

In [ ]:
# ── Change 1: keep side-information columns (NOT model features) ──────────
META_SOURCE = {
    '_STATE'  : 'meta_state',     # US state / territory code
    '_LLCPWT' : 'meta_weight',    # final survey weight
    '_STSTR'  : 'meta_strata',    # sampling stratum
    '_PSU'    : 'meta_psu',       # primary sampling unit
    '_RACEGR3': 'meta_race',      # race/ethnicity: 1=White NH, 2=Black NH, 3=Other NH,
                                  # 4=Multiracial NH, 5=Hispanic, 9=Don't know/Refused
}

for src, new in META_SOURCE.items():
    df[new] = df[src]

META_COLS = list(META_SOURCE.values())
print("✓ Side-information columns kept:")
for src, new in META_SOURCE.items():
    print(f"   {src:<9} → {new:<12} non-missing: {df[new].notna().sum():,}")

In [ ]:
# ── Define all columns to drop ────────────────────────────────────────────
# Change 2: this is now the ONLY drop cell (v1 had a duplicate that dropped nothing)
ADMIN = [
    # Record identification
    '_STATE','FMONTH','IDATE','IMONTH','IDAY','IYEAR','DISPCODE','SEQNO','_PSU',
    # Landline introduction
    'CTELENM1','PVTRESD1','COLGHOUS','STATERE1','CELPHON1','LADULT1','NUMADULT',
    'RESPSLC1','LANDSEX2','LNDSXBRT',
    # Cell phone introduction
    'SAFETIME','CTELNUM1','CELLFON5','CADULT1','CELLSEX2','CELSXBRT',
    'PVTRESD3','CCLGHOUS','CSTATE1','LANDLINE','HHADULT',
    # Questionnaire admin
    'QSTVER','QSTLANG',
    # Sampling weights (copied to meta_* above)
    'MSCODE','_STSTR','_STRWT','_RAWRAKE','_WT2RAKE','_IMPRACE',
    '_DUALUSE','_DUALCOR','_LLCPWT2','_LLCPWT',
    # Child variables
    'RCSBORG1','RCSRLTN2','RCSGEND1','RCSXBRTH',
    'CASTHDX2','CASTHNO2','CAGEG','_CLLCPWT','_CHISPNC','_CRACE1',
]

CALCULATED = [
    '_RFHLTH','_PHYS14D','_MENT14D','_HLTHPL1','_HCVU653',
    '_TOTINDA','METVL12_','METVL22_','MAXVO21_','FC601_',
    'ACTIN13_','ACTIN23_','PADUR1_','PADUR2_','PAFREQ1_','PAFREQ2_',
    '_MINAC12','_MINAC22','STRFREQ_','PAMISS3_','PAMIN13_','PAMIN23_',
    'PA3MIN_','PAVIG13_','PAVIG23_','PA3VIGM_',
    '_PACAT3','_PAINDX3','_PA150R4','_PA300R4','_PA30023',
    '_PASTRNG','_PAREC3','_PASTAE3',
    '_RFHYPE6','_CHOLCH3','_RFCHOL3','_MICHD',
    '_LTASTH1','_CASTHM1','_ASTHMS1','_DRDXAR2',
    '_MRACE1','_HISPANC','_RACE','_RACEG21','_RACEGR3','_RACEPRV',
    '_SEX','_AGEG5YR','_AGE65YR','_AGE_G',
    'HTIN4','HTM4','WTKG3',
    '_BMI5CAT','_RFBMI5',
    '_CHLDCNT','_EDUCAG','_INCOMG1',
    '_SMOKER3','_RFSMOK3','_CURECI2',
    'DRNKANY6','DROCDY4_','_RFBING6','_DRNKWK2','_RFDRHV8',
    '_FLSHOT7','_PNEUMO3','_AIDTST4','_RFSEAT2','_RFSEAT3','_DRNKDRV',
    '_METSTAT','_URBSTAT',
]

REDUNDANT = ['HEIGHT3', 'WEIGHT2']

NON_CLINICAL = [
    'NUMHHOL4','NUMPHON4','CPDEMO1C',
    'BIRTHSEX','TRNSGNDR','USEMRJN4',
]

# Change 2: report each group separately so the paper's counts add up exactly
print(f"{'Group':<14} {'Listed':>7} {'In file':>8} {'Not in file':>12}")
for name, group in [('ADMIN', ADMIN), ('CALCULATED', CALCULATED),
                    ('REDUNDANT', REDUNDANT), ('NON_CLINICAL', NON_CLINICAL)]:
    present = [c for c in group if c in df.columns]
    missing = [c for c in group if c not in df.columns]
    print(f"{name:<14} {len(group):>7} {len(present):>8} {len(missing):>12}  {missing if missing else ''}")

ALL_DROP = ADMIN + CALCULATED + REDUNDANT + NON_CLINICAL

cols_before = df.shape[1]
to_drop   = [c for c in ALL_DROP if c in df.columns]
not_found = [c for c in ALL_DROP if c not in df.columns]

df = df.drop(columns=to_drop)

print()
print(f"Columns before  : {cols_before}  (350 raw + {len(META_COLS)} meta)")
print(f"Columns dropped : {len(to_drop)}")
print(f"Not found       : {len(not_found)} {not_found}")
print(f"Columns after   : {df.shape[1]}  ({df.shape[1] - len(META_COLS)} + {len(META_COLS)} meta)")

log_step('After dropping admin / calculated / redundant / non-clinical columns',
         f'{len(to_drop)} columns dropped; {len(not_found)} listed but absent')

### Step: Encode Stroke Target Variable (CVDSTRK3)

CVDSTRK3 asks whether the respondent was **ever told** they had a stroke (prevalent, self-reported stroke history).
Original coding: 1=Yes, 2=No, 7=Don't know, 9=Refused. We remap to 1=Stroke, 0=No stroke; 7/9/blank → dropped (no target).

In [ ]:
# ── Encode Target Variable (CVDSTRK3 → stroke) ───────────────────────────
print("=== CVDSTRK3 BEFORE ENCODING ===")
print(df['CVDSTRK3'].value_counts(dropna=False).sort_index())

stroke1_before = int((df['CVDSTRK3'] == 1.0).sum())

df['CVDSTRK3'] = df['CVDSTRK3'].map({
    1.0: 1,       # Yes had stroke
    2.0: 0,       # No stroke
    7.0: np.nan,  # Don't know → cannot use as target
    9.0: np.nan   # Refused   → cannot use as target
})

rows_before = len(df)
df = df.dropna(subset=['CVDSTRK3'])
rows_dropped = rows_before - len(df)

df = df.rename(columns={'CVDSTRK3': 'stroke'})

stroke1_after = int((df['stroke'] == 1).sum())
stroke0_after = int((df['stroke'] == 0).sum())

print(f"\n=== AFTER ENCODING ===")
print(f"Rows dropped (no answer): {rows_dropped:,}")
print(f"Rows remaining          : {len(df):,}")
print(f"Stroke=1 (had stroke)   : {stroke1_after:,}")
print(f"Stroke=0 (no stroke)    : {stroke0_after:,}")
print(f"Stroke prevalence       : {stroke1_after/len(df)*100:.2f}%")

if stroke1_before == stroke1_after:
    print("✓ All stroke=1 patients preserved")
else:
    print("✗ WARNING — stroke=1 count changed — investigate immediately")

log_step('After removing respondents with unknown stroke answer (7/9/blank)',
         f'{rows_dropped:,} rows dropped')

### Step: Fix _BMI5 (Body Mass Index)

Missing code 99999 is replaced with NaN **before** dividing by 100 (BRFSS stores BMI × 100).

In [ ]:
# ── Fix _BMI5 ─────────────────────────────────────────────────────────────
print("=== _BMI5 BEFORE FIX ===")
print(f"Min : {df['_BMI5'].min()}")
print(f"Max : {df['_BMI5'].max()}")
print(f"NaN : {df['_BMI5'].isna().sum():,}")

df['_BMI5'] = df['_BMI5'].replace(99999.0, np.nan)
df['_BMI5'] = df['_BMI5'] / 100
df = df.rename(columns={'_BMI5': 'bmi'})

bmi_valid = df['bmi'].dropna()
print(f"\n=== bmi AFTER FIX ===")
print(f"Min  : {bmi_valid.min():.2f}")
print(f"Max  : {bmi_valid.max():.2f}")
print(f"Mean : {bmi_valid.mean():.2f}")
print(f"NaN  : {df['bmi'].isna().sum():,}")

if bmi_valid.max() >= 999:
    print("✗ WARNING — max BMI >= 999 — 99999 fix did not work")
elif bmi_valid.min() < 5:
    print("✗ WARNING — min BMI < 5 — check for remaining invalid values")
else:
    print("✓ BMI values look correct")

### Step: Fix _AGE80 and SEXVAR

_AGE80: age capped at 80 by CDC for privacy — rename to `age`. SEXVAR: 1=Male, 2=Female → 1/0, rename to `gender`.

In [ ]:
# ── Fix _AGE80 → age ──────────────────────────────────────────────────────
print("=== _AGE80 BEFORE RENAME ===")
print(f"Min  : {df['_AGE80'].min()}")
print(f"Max  : {df['_AGE80'].max()}")
print(f"Mean : {df['_AGE80'].mean():.1f}")
print(f"NaN  : {df['_AGE80'].isna().sum():,}")
df = df.rename(columns={'_AGE80': 'age'})
print("✓ _AGE80 renamed to age")

# ── Fix SEXVAR → gender ───────────────────────────────────────────────────
print("\n=== SEXVAR BEFORE REMAP ===")
print(df['SEXVAR'].value_counts(dropna=False).sort_index())
df['SEXVAR'] = df['SEXVAR'].map({1.0: 1, 2.0: 0})
df = df.rename(columns={'SEXVAR': 'gender'})
print("\n=== gender AFTER REMAP ===")
print(df['gender'].value_counts(dropna=False).sort_index())
print("✓ SEXVAR remapped and renamed to gender")

### Step: Convert Special Missing Codes to NaN

Every rule below was checked **automatically** against the official BRFSS 2023 codebook
(USCODE23_LLCP_021924.HTML): each code we turn into NaN is labelled "Don't know/Not sure" or "Refused" for that
variable, and every kept variable that has such codes is covered by a rule. Variables are listed by code family
(7/9, 77/99, 777/999, 777777/999999, 98/99, special cases).
**v2 fix:** `EMPLOY1` was in the 7/9 list, but for this question 7 = **Retired** (a real answer); only 9 = Refused is now removed.
88 means "none/zero" in 11 variables and is recoded to 0, not NaN.
Excluded: `stroke`, `bmi`, `age`, `gender` (done above), `BPHIGH6`, `DIABETE4` (next steps), `ALCDAY4` (composite decode) and all `meta_*` columns.

In [ ]:
# ── Convert Special Missing Codes to NaN ──────────────────────────────────
# Every rule checked automatically against the official BRFSS 2023 codebook
#   7/9           — 163 variables
#   77/99         — 23 variables
#   777/999       — 8 exercise / duration variables
#   777777/999999 — 2 date variables
#   98/99         — 2 age variables
#   special cases — CNCRTYP2, CSRVTRT3, TETANUS1, EMPLOY1 (9 only), CHILDREN (99 only)
#   88 = None/Zero, NOT missing — 11 variables (recode to 0)

nan_before = df.isnull().sum().sum()

# ── Part A: 7/9 missing codes ─────────────────────────────────────────────
standard_7_9 = [
    'ACEADNED','ACEADSAF','ACEDEPRS','ACEDIVRC','ACEDRINK','ACEDRUGS',
    'ACEHURT1','ACEHVSEX','ACEPRISN','ACEPUNCH','ACESWEAR','ACETOUCH',
    'ACETTHEM','ADDEPEV3','ARTHDIS2','ARTHEDU','ARTHEXER','ASPIRIN',
    'ASTHMA3','ASTHNOW','BLDSTFIT','BLIND','BPMEDS1','CAREGIV1',
    'CDDISCU1','CDHOUS1','CDSOCIA1','CDWORRY','CERVSCRN','CHCCOPD3',
    'CHCKDNY2','CHCOCNC1','CHCSCNC1','CHECKUP1','CHOLCHK3','CHOLMED3',
    'CIMEMLO1','CNCRDIFF','COLNCNCR','COLNSIGM','COLNTES1','COVACGE1',
    'COVIDACT','COVIDNU2','COVIDPO1','COVIDSM1','COVIDVA1','CRGVALZD',
    'CRGVEXPT','CRGVHOU1','CRGVHRS1','CRGVLNG1','CRGVPER1','CRVCLCNC',
    'CRVCLHPV','CRVCLPAP','CSRVCLIN','CSRVCTL2','CSRVDEIN','CSRVINSR',
    'CSRVINST','CSRVPAIN','CSRVRTRN','CSRVSUM','CVDCRHD4','CVDINFR4',
    'DEAF','DECIDE','DIABEDU1','DIABEYE1','DIABTYPE','DIFFALON',
    'DIFFDRES','DIFFWALK','ECIGNOW2','EDUCA','EMTSUPRT',
    'EXERANY2','EYEEXAM1','FEETSORE','FIREARM5','FIRSTAID','FLUSHOT7',
    'FOODSTMP','GENHLTH','GUNLOAD','HADHYST2','HADMAM','HADSIGM4',
    'HASYMP1','HASYMP2','HASYMP3','HASYMP4','HASYMP5','HASYMP6',
    'HAVARTH4','HEATTBCO','HIVTST7','HOWLONG','HPVADVC4','INSULIN1',
    'LASTSIG4','LCSCTSC1','LCSCTWHN','LCSSCNCR','LMTJOIN3','LOADULK2',
    'LSATISFY','MARITAL','MARJDAB','MARJEAT','MARJOTHR','MARJSMOK',
    'MARJVAPE','MEDCOST1','MENTCIGS','MENTECIG','PCPSARS2','PCSTALK2',
    'PDIABTS1','PERSDOC3','PNEUVAC4','PREDIAB2','PREGNANT','PSASUGS1',
    'PSATEST1','PSATIME1','RENTHOM1','RRATWRK2','RRCOGNT2','RRHCARE4',
    'RRPHYSM2','RRTREAT','SDHBILLS','SDHEMPLY','SDHFOOD1','SDHSTRE1',
    'SDHTRNSP','SDHUTILS','SDLONELY','SDNATES1','SEATBELT','SHINGLE2',
    'SIGMTES1','SMALSTOL','SMOKDAY2','SMOKE100','SOFEMALE','SOMALE',
    'STOLTEST','STOOLDN2','STOPSMK2','STRSYMP1','STRSYMP2','STRSYMP3',
    'STRSYMP4','STRSYMP5','STRSYMP6','TOLDHI3','USENOW3','VCLNTES2',
    'VETERAN3','VIRCOLO1',
]
for col in standard_7_9:
    if col in df.columns:
        df[col] = df[col].replace({7.0: np.nan, 9.0: np.nan})
print(f"✓ Part A — 7/9 replaced in {sum(1 for c in standard_7_9 if c in df.columns)} columns")

# ── Part B: 77/99 missing codes ───────────────────────────────────────────
standard_77_99 = [
    'AVEDRNK3','CHKHEMO3','CRGVPRB3','CRGVREL4',
    'CSRVDOC1','DRNK3GE5','DRNKDRI2','EXRACT12','EXRACT22',
    'FALL12MN','FALLINJ5','HPVADSHT','IMFVPLA4','INCOME3',
    'JOINPAI2','LASTSMK2','MARIJAN1','MAXDRNKS','MENTHLTH',
    'PHYSHLTH','POORHLTH','PRIMINS1','RRCLASS3',
]
for col in standard_77_99:
    if col in df.columns:
        df[col] = df[col].replace({77.0: np.nan, 99.0: np.nan})
print(f"✓ Part B — 77/99 replaced in {sum(1 for c in standard_77_99 if c in df.columns)} columns")

# ── Part C: 777/999 missing codes ─────────────────────────────────────────
codes_777_999 = [
    'EXERHMM1','EXERHMM2','EXEROFT1','EXEROFT2',
    'LCSFIRST','LCSLAST','LCSNUMCG','STRENGTH',
]
for col in codes_777_999:
    if col in df.columns:
        df[col] = df[col].replace({777.0: np.nan, 999.0: np.nan})
print(f"✓ Part C — 777/999 replaced in {sum(1 for c in codes_777_999 if c in df.columns)} columns")

# ── Part D: 777777/999999 missing codes (date variables, MMYYYY) ──────────
codes_777777 = ['FLSHTMY3', 'HIVTSTD3']
for col in codes_777777:
    if col in df.columns:
        df[col] = df[col].replace({777777.0: np.nan, 999999.0: np.nan})
print(f"✓ Part D — 777777/999999 replaced in {sum(1 for c in codes_777777 if c in df.columns)} columns")

# ── Part E: 98/99 missing codes (age-at-diagnosis variables) ──────────────
codes_98_99 = ['DIABAGE4', 'CNCRAGE']
for col in codes_98_99:
    if col in df.columns:
        df[col] = df[col].replace({98.0: np.nan, 99.0: np.nan})
print(f"✓ Part E — 98/99 replaced in {sum(1 for c in codes_98_99 if c in df.columns)} columns")

# ── Part F: special cases ─────────────────────────────────────────────────
if 'CNCRTYP2' in df.columns:   # 1-23 real types; 77/99 missing
    df['CNCRTYP2'] = df['CNCRTYP2'].replace({77.0: np.nan, 99.0: np.nan})
if 'CSRVTRT3' in df.columns:   # 1-3 real; 7/9 missing
    df['CSRVTRT3'] = df['CSRVTRT3'].replace({7.0: np.nan, 9.0: np.nan})
if 'TETANUS1' in df.columns:   # 1-3 real; 7/9 missing
    df['TETANUS1'] = df['TETANUS1'].replace({7.0: np.nan, 9.0: np.nan})
if 'EMPLOY1' in df.columns:    # v2 fix: 7 = Retired (real answer); only 9 = Refused is missing
    df['EMPLOY1'] = df['EMPLOY1'].replace({9.0: np.nan})
if 'CHILDREN' in df.columns:   # 1-87 = number of children, 88 = none; only 99 = Refused is missing
    df['CHILDREN'] = df['CHILDREN'].replace({99.0: np.nan})
print("✓ Part F — special cases handled (CNCRTYP2, CSRVTRT3, TETANUS1, EMPLOY1, CHILDREN)")
print(f"   EMPLOY1 = 7 (Retired) kept: {int((df['EMPLOY1'] == 7).sum()):,} respondents")

# ── Part G: 88 = None → 0 (NOT missing) ───────────────────────────────────
cols_88_none = [
    'PHYSHLTH','MENTHLTH','POORHLTH','CHILDREN','FALL12MN','FALLINJ5',
    'AVEDRNK3','DRNK3GE5','DRNKDRI2','CHKHEMO3','MARIJAN1',
]
for col in cols_88_none:
    if col in df.columns:
        df[col] = df[col].replace({88.0: 0})
print(f"✓ Part G — 88=None recoded to 0 in {sum(1 for c in cols_88_none if c in df.columns)} columns")

nan_after = df.isnull().sum().sum()
print(f"\nNaN cells before : {nan_before:,}")
print(f"NaN cells after  : {nan_after:,}")
print(f"New NaN created  : {nan_after - nan_before:,}")
print(f"Dataset shape    : {df.shape[0]:,} rows x {df.shape[1]} columns")

### Step: Remap Binary Columns (2=No → 0=No)

Only columns whose remaining non-missing values are exactly {1, 2} (or just one of them) are remapped.
`meta_*` columns are never touched.

In [ ]:
# ── Remap Binary Columns (2=No → 0=No) ───────────────────────────────────
EXCLUDE_BINARY = {'stroke', 'bmi', 'age', 'gender', 'BPHIGH6', 'DIABETE4'}

remapped_cols = []
for col in df.columns:
    if col in EXCLUDE_BINARY or col.startswith('meta_'):   # v2: never touch meta_*
        continue
    unique_vals = set(df[col].dropna().unique())
    if unique_vals in ({1.0, 2.0}, {1.0}, {2.0}):
        df[col] = df[col].replace({2.0: 0})
        remapped_cols.append(col)

print(f"Binary columns remapped (2→0) : {len(remapped_cols)}")
print(remapped_cols)

print("\n=== SPOT CHECK ===")
for col in ['SMOKE100', 'EXERANY2', 'CVDCRHD4', 'ADDEPEV3']:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).sort_index())

### Change 3 — Hypertension and diabetes, using CDC definitions

**v1 problems:** pregnancy-only and borderline high blood pressure were counted as hypertension, and diabetes was
turned into an invented glucose value (90 / 115 / 180 mg/dL) called `avg_glucose`, which looked like a real lab measurement.

**v2 rules (following the CDC's own calculated variables in the 2023 codebook):**

| BPHIGH6 answer | v1 | v2 `hypertension` |
|---|---|---|
| 1 = Yes | 1 | **1** |
| 2 = Yes, only during pregnancy | 1 | **0** |
| 3 = No | 0 | **0** |
| 4 = Borderline / pre-hypertensive | 1 | **0** |

(Same definition as the CDC's `_RFHYPE6`: only answer 1 counts as hypertension.)

| DIABETE4 answer | v1 `avg_glucose` | v2 `diabetes_status` |
|---|---|---|
| 3 = No | 90 | **0** = no diabetes |
| 2 = Yes, only during pregnancy | 180 | **0** = no diabetes |
| 4 = Pre-diabetes / borderline | 115 | **1** = pre-diabetes |
| 1 = Yes | 180 | **2** = diabetes |

`diabetes_status` is an ordered category (0 < 1 < 2), not a measurement. 7/9 → missing.

In [ ]:
# ── Change 3a: BPHIGH6 → hypertension (CDC _RFHYPE6 definition) ──────────
print("=== BPHIGH6 BEFORE MAPPING ===")
print(df['BPHIGH6'].value_counts(dropna=False).sort_index())

df['BPHIGH6'] = df['BPHIGH6'].map({
    1.0: 1,   # Yes — told they have high blood pressure
    2.0: 0,   # Yes, but only during pregnancy → not hypertensive
    3.0: 0,   # No
    4.0: 0,   # Borderline / pre-hypertensive → not hypertensive
})            # 7 / 9 / blank → NaN
df = df.rename(columns={'BPHIGH6': 'hypertension'})

print("\n=== hypertension AFTER MAPPING ===")
print(df['hypertension'].value_counts(dropna=False).sort_index())
print("✓ BPHIGH6 → hypertension done")

# ── Change 3b: DIABETE4 → diabetes_status (ordinal 0/1/2) ────────────────
print("\n=== DIABETE4 BEFORE MAPPING ===")
print(df['DIABETE4'].value_counts(dropna=False).sort_index())

df['DIABETE4'] = df['DIABETE4'].map({
    3.0: 0,   # No
    2.0: 0,   # Yes, but only during pregnancy (gestational) → no diabetes
    4.0: 1,   # Pre-diabetes / borderline
    1.0: 2,   # Yes — diabetes
})            # 7 / 9 / blank → NaN
df = df.rename(columns={'DIABETE4': 'diabetes_status'})

print("\n=== diabetes_status AFTER MAPPING ===")
print(df['diabetes_status'].value_counts(dropna=False).sort_index())
print("✓ DIABETE4 → diabetes_status done (0=no, 1=pre-diabetes, 2=diabetes)")

### Step: Fix ALCDAY4 (Alcohol Days)

1xx = days per week (×4.3 to monthly), 2xx = days per month, 888 = none → 0, 777/999 → missing.

In [ ]:
# ── Fix ALCDAY4 composite encoding ───────────────────────────────────────
print("=== ALCDAY4 BEFORE FIX ===")
print(f"Unique values : {df['ALCDAY4'].nunique()}")
print(f"NaN           : {df['ALCDAY4'].isna().sum():,}")

def decode_alcday4(val):
    """
    Convert ALCDAY4 composite code to drinking days per month.
    1xx = per week (x 4.3), 2xx = per month, 888 = none, else NaN
    """
    if pd.isna(val):
        return np.nan
    val = int(val)
    if val == 888:
        return 0
    elif 101 <= val <= 107:
        return round((val - 100) * 4.3)
    elif 201 <= val <= 230:
        return val - 200
    else:
        return np.nan   # 777 / 999 / unexpected

df['ALCDAY4'] = df['ALCDAY4'].apply(decode_alcday4)
df = df.rename(columns={'ALCDAY4': 'alcohol_days_per_month'})

print("\n=== alcohol_days_per_month AFTER FIX ===")
print(f"Min  : {df['alcohol_days_per_month'].min()}")
print(f"Max  : {df['alcohol_days_per_month'].max()}")
print(f"Mean : {df['alcohol_days_per_month'].mean():.1f}")
print(f"NaN  : {df['alcohol_days_per_month'].isna().sum():,}")
print("✓ ALCDAY4 decoded to alcohol_days_per_month")

### Change 4 — No outcome-dependent row filtering

**v1** dropped non-stroke respondents with >70% missing values but kept every stroke respondent.
That made "mostly blank" a hidden signal for stroke and raised prevalence from ~4.25% to 6.17% (reviewer comment #2).

**v2** keeps every respondent with a known stroke answer. Instead it:
1. saves each respondent's share of missing answers as `meta_missing_pct`, so later notebooks can apply **one** threshold to **both** classes, and run the sensitivity analysis the reviewer asked for;
2. prints what each threshold would remove, separately for stroke and non-stroke;
3. drops only columns that are completely empty (they carry no information).

In [ ]:
# ── Change 4: missingness summary instead of one-sided row filtering ──────
feature_cols = [c for c in df.columns if c != 'stroke' and not c.startswith('meta_')]

# Columns with no values at all carry no information — drop and log them
all_empty = [c for c in feature_cols if df[c].isna().all()]
if all_empty:
    df = df.drop(columns=all_empty)
    feature_cols = [c for c in feature_cols if c not in all_empty]
print(f"Completely empty columns dropped: {len(all_empty)} {all_empty}")

# Share of missing answers per respondent (features only)
df['meta_missing_pct'] = df[feature_cols].isna().mean(axis=1)

print(f"\nFeature columns : {len(feature_cols)}")
print(f"Respondents     : {len(df):,}")
print(f"Overall missing : {df[feature_cols].isna().mean().mean()*100:.2f}% of feature cells")

# What a SYMMETRIC threshold would remove (same rule for both classes)
print(f"\n{'Threshold':>10} {'Removed stroke=1':>17} {'Removed stroke=0':>17} "
      f"{'Rows left':>10} {'Prevalence left':>16}")
for t in [None, 0.9, 0.7, 0.5]:
    if t is None:
        keep = pd.Series(True, index=df.index)
        label = 'none'
    else:
        keep = df['meta_missing_pct'] <= t
        label = f'>{int(t*100)}%'
    removed = df.loc[~keep, 'stroke']
    kept    = df.loc[keep, 'stroke']
    print(f"{label:>10} {int((removed==1).sum()):>17,} {int((removed==0).sum()):>17,} "
          f"{len(kept):>10,} {kept.mean()*100:>15.2f}%")

print("\n✓ No rows dropped here — filtering (if any) is applied later, identically to both classes")
log_step('Final cleaned dataset (no outcome-dependent filtering)',
         f'{len(all_empty)} empty columns dropped; meta_missing_pct added')

### Change 5 — Save cleaned dataset and cleaning flow table

In [ ]:
# ── Save cleaned dataset ──────────────────────────────────────────────────
df.to_csv(CLEANED_PATH, index=False)
file_size_mb = os.path.getsize(CLEANED_PATH) / (1024**2)

# ── Save flow table (numbers for the paper's Table 9-2 / Fig. 9-2) ───────
flow_df = pd.DataFrame(flow)
flow_df.to_csv(FLOW_PATH, index=False)

n_stroke    = int((df['stroke'] == 1).sum())
n_no_stroke = int((df['stroke'] == 0).sum())
meta_cols   = [c for c in df.columns if c.startswith('meta_')]
n_features  = df.shape[1] - 1 - len(meta_cols)

print("=" * 60)
print("  BRFSS 2023 — Cleaning Complete (v2)")
print("=" * 60)
print(flow_df.to_string(index=False))
print()
print(f"  Respondents       : {len(df):,}")
print(f"  Model features    : {n_features}")
print(f"  Target            : stroke")
print(f"  Side columns      : {len(meta_cols)} {meta_cols}")
print(f"  Stroke=1          : {n_stroke:,} ({n_stroke/len(df)*100:.2f}%)")
print(f"  Stroke=0          : {n_no_stroke:,} ({n_no_stroke/len(df)*100:.2f}%)")
print()
print(f"  Saved data : {CLEANED_PATH} ({file_size_mb:.1f} MB)")
print(f"  Saved flow : {FLOW_PATH}")
print("=" * 60)

key_cols = ['stroke', 'bmi', 'age', 'gender', 'hypertension',
            'diabetes_status', 'alcohol_days_per_month',
            'SMOKE100', 'EXERANY2', 'CVDCRHD4', 'meta_race', 'meta_weight']
print("\n=== KEY COLUMNS CHECK ===")
for col in key_cols:
    print(f"  {'✓' if col in df.columns else '✗ MISSING'} {col}")

### Note on variable types (for the modelling notebooks)

- **Nominal** (no natural order — must be one-hot encoded, never averaged): MARITAL, EMPLOY1, PRIMINS1, SOMALE, SOFEMALE,
  RRCLASS3, CSRVDOC1, FIRSTAID, RRTREAT, RRATWRK2, RRHCARE4, ECIGNOW2, TETANUS1, ASPIRIN, COVACGE1, CSRVTRT3, CSRVCTL2, PCSTALK2
- **Ordinal** (ordered codes): e.g. GENHLTH, EDUCA, INCOME3, CHECKUP1, `diabetes_status`
- **Binary** (0/1): the remapped yes/no columns, `hypertension`, `gender`
- **Continuous**: `age`, `bmi`, `alcohol_days_per_month`, day counts such as PHYSHLTH / MENTHLTH
- **`meta_*`**: side information only — drop before training